# 14 — Prompt Evaluation

## Scenario
Northstar Support Copilot receives a variety of customer tickets. We need to evaluate two types of prompts:
1. **Deterministic Prompts:** Routing tickets to the correct department (`Refund`, `Technical Support`, or `Other`).
2. **Generative Prompts:** Drafting email responses to customers.

**The Problem:** How do we know if changing a prompt makes it "better"? Anecdotal testing ("vibe checks") doesn't scale. We need programmatic ways to evaluate prompt performance over frozen datasets.

In [ ]:
import os
from enum import Enum
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Initialize the client
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'


---
## Part 1: Deterministic Evaluation (Classification)

For tasks like routing or extraction, we use Pydantic to ensure the model's output is strictly typed. This makes automated grading trivial (exact match).

In [ ]:
# 1. Define the Schema
class TicketCategory(str, Enum):
    REFUND = "Refund"
    TECHNICAL = "Technical Support"
    OTHER = "Other"

class TicketClassification(BaseModel):
    category: TicketCategory

# 2. Define the "Golden" Dataset
classification_dataset = [
    {"ticket": "My mug arrived broken, I want my money back.", "expected": "Refund"},
    {"ticket": "The website keeps crashing when I try to log in.", "expected": "Technical Support"},
    {"ticket": "What time does the store open?", "expected": "Other"},
    # Edge case: Mentioning the word 'refund' but not actually asking for one
    {"ticket": "I know your refund policy is strict, but my mug's handle just snapped off after normal use. How do I fix it?", "expected": "Technical Support"},
]


In [ ]:
baseline_prompt_template = """\nClassify the following customer ticket.\n\nTicket: {ticket}\n"""

def evaluate_classification_prompt(prompt_template, dataset):
    correct = 0
    for i, data in enumerate(dataset):
        ticket = data["ticket"]
        expected = data["expected"]
        
        prompt = prompt_template.format(ticket=ticket)
        
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=TicketClassification,
            )
        )
        
        # The model returns JSON matching the Pydantic schema
        classification = TicketClassification.model_validate_json(response.text)
        actual = classification.category.value
        
        is_match = (actual == expected)
        if is_match:
            correct += 1
            
        print(f"[{'PASS' if is_match else 'FAIL'}] Ticket {i+1}: Expected '{expected}', Got '{actual}'")
        
    accuracy = (correct / len(dataset)) * 100
    print(f"\nTotal Accuracy: {accuracy:.1f}%")
    return accuracy

print("--- BASELINE EVALUATION ---")
baseline_accuracy = evaluate_classification_prompt(baseline_prompt_template, classification_dataset)


### Fixing the Regression
The baseline likely fails the edge case because it sees the word "refund" and jumps to conclusions. We update our prompt (Candidate) with better instructions and re-run the *exact same evaluation loop*.

In [ ]:
candidate_prompt_template = """\nYou are an expert customer service router for Northstar.\nClassify the following customer ticket.\n\nCRITICAL RULE: Do not classify a ticket as a 'Refund' just because it contains the word refund. \nOnly classify it as a 'Refund' if the user is explicitly requesting their money back. \nIf they are asking for help fixing something, it is 'Technical Support'.\n\nTicket: {ticket}\n"""

print("\n--- CANDIDATE EVALUATION ---")
candidate_accuracy = evaluate_classification_prompt(candidate_prompt_template, classification_dataset)
print(f"\nImprovement: +{candidate_accuracy - baseline_accuracy}% accuracy")


---
## Part 2: LLM-as-a-Judge (Generative Evaluation)

For generative tasks (like drafting an email), there is no single "correct" answer, so exact-match deterministic checks fail. Instead, we use a strong LLM to "judge" the output against a specific rubric.

In [ ]:
# 1. The Generative Task
drafting_prompt = """
Write a short response to this customer email. Be helpful.
Customer: "I can't log into my account, it says 'password incorrect' but I swear it's right!"
"""

draft_response = client.models.generate_content(
    model=MODEL_ID,
    contents=drafting_prompt,
    config=types.GenerateContentConfig(temperature=0.7)
)
generated_email = draft_response.text

print("--- GENERATED EMAIL ---")
print(generated_email)


### Building the Judge
We define a strict rubric using Pydantic, and ask the model to evaluate the generated email.

In [ ]:
class JudgeRubric(BaseModel):
    is_polite: bool = Field(description="Does the email sound empathetic and polite?")
    offers_solution: bool = Field(description="Does it suggest resetting the password?")
    score: int = Field(description="Overall quality score from 1 to 5.")
    reasoning: str = Field(description="Brief explanation for the score.")

def evaluate_generation_with_judge(email_text: str) -> JudgeRubric:
    judge_prompt = f"""
    You are an expert QA evaluator for Northstar Customer Support.
    Evaluate the following generated support email against our quality rubric.
    
    Generated Email:
    <email>
    {email_text}
    </email>
    """
    
    response = client.models.generate_content(
        model=MODEL_ID, # In production, the Judge model is often a larger/more capable model than the generator.
        contents=judge_prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=JudgeRubric,
        )
    )
    
    return JudgeRubric.model_validate_json(response.text)

print("\n--- LLM-AS-A-JUDGE EVALUATION ---")
evaluation = evaluate_generation_with_judge(generated_email)
print(f"Polite? {evaluation.is_polite}")
print(f"Offers Solution? {evaluation.offers_solution}")
print(f"Score: {evaluation.score}/5")
print(f"Reasoning: {evaluation.reasoning}")


## Conclusion

- **Deterministic Tasks (Routing/Extraction):** Use Pydantic schemas and exact-match string comparisons.
- **Generative Tasks (Drafting/Summarizing):** Use a Judge Rubric (also powered by Pydantic) to evaluate qualitative metrics like politeness or accuracy.

By defining golden datasets and building these automated loops, you can implement **Continuous Evaluation (PromptOps)** in your CI/CD pipelines, mathematically proving that your prompt changes are actually improvements.